# YgoMaster-PackBuilder — Pack Assembly Tests

Notebook offline per il single-pack assembler aggiornato al nuovo `RarityMapping`.

Verifica:
- main, mini e shared pack;
- pack rules;
- opening-policy inheritance e override;
- YDK -> Master Duel ID;
- unresolved YDK IDs;
- `Secret Rare -> N` per main/mini;
- `Secret Rare -> UR` per shared staples;
- nuova composizione staples `100/60/20/20`;
- mismatch dell'external ID;
- source rarity sconosciuta.


## 1. Repository root

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.assembly import assemble_pack_definition
from ygomaster_packbuilder.domain import PackDefinition, PackType, Rarity, SourceProvider
from ygomaster_packbuilder.manifest.models import (
    ManifestPack,
    ManifestPackSource,
    ManifestSharedPack,
)
from ygomaster_packbuilder.mapping import RarityMapping, load_rarity_mapping
from ygomaster_packbuilder.rules import get_pack_rule
from ygomaster_packbuilder.sources.ygoprodeck import (
    YgoProDeckPackCard,
    YgoProDeckPackData,
)

print("Imports OK")

Imports OK


## 3. Mapping reale e YDK mapping di test

In [3]:
rarity_mapping = load_rarity_mapping(
    repo_root / "config" / "rarity_mapping.yaml"
)
assert isinstance(rarity_mapping, RarityMapping)

ydk_mapping = {
    1001: 501,
    1002: 502,
    1003: 503,
    1004: 504,
    1005: 505,
}

print(rarity_mapping)
print(ydk_mapping)

RarityMapping(rarities={'Common': <Rarity.N: 'N'>, 'Rare': <Rarity.R: 'R'>, 'Super Rare': <Rarity.SR: 'SR'>, 'Ultra Rare': <Rarity.UR: 'UR'>}, overrides={<PackType.PERSONAL_MAIN: 'personal_main'>: {'Secret Rare': <Rarity.N: 'N'>}, <PackType.PERSONAL_MINI: 'personal_mini'>: {'Secret Rare': <Rarity.N: 'N'>}, <PackType.SHARED_STAPLES: 'shared_staples'>: {'Secret Rare': <Rarity.UR: 'UR'>}})
{1001: 501, 1002: 502, 1003: 503, 1004: 504, 1005: 505}


## 4. Source pack offline

In [4]:
source_pack = YgoProDeckPackData(
    external_id="24116",
    name="Scaled Invasion",
    description="Offline assembly test pack",
    creator="livius03",
    image_id="8400623",
    cards=[
        YgoProDeckPackCard(
            name="Card N",
            ydk_id=1001,
            card_type="Effect Monster",
            source_rarity="Common",
        ),
        YgoProDeckPackCard(
            name="Card R",
            ydk_id=1002,
            card_type="Spell Card",
            source_rarity="Rare",
        ),
        YgoProDeckPackCard(
            name="Card SR",
            ydk_id=1003,
            card_type="Trap Card",
            source_rarity="Super Rare",
        ),
        YgoProDeckPackCard(
            name="Card UR",
            ydk_id=1004,
            card_type="Effect Monster",
            source_rarity="Ultra Rare",
        ),
        YgoProDeckPackCard(
            name="Card Secret",
            ydk_id=1005,
            card_type="Effect Monster",
            source_rarity="Secret Rare",
        ),
    ],
)

print(source_pack)

YgoProDeckPackData(external_id='24116', name='Scaled Invasion', description='Offline assembly test pack', creator='livius03', image_id='8400623', cards=[YgoProDeckPackCard(name='Card N', ydk_id=1001, card_type='Effect Monster', source_rarity='Common'), YgoProDeckPackCard(name='Card R', ydk_id=1002, card_type='Spell Card', source_rarity='Rare'), YgoProDeckPackCard(name='Card SR', ydk_id=1003, card_type='Trap Card', source_rarity='Super Rare'), YgoProDeckPackCard(name='Card UR', ydk_id=1004, card_type='Effect Monster', source_rarity='Ultra Rare'), YgoProDeckPackCard(name='Card Secret', ydk_id=1005, card_type='Effect Monster', source_rarity='Secret Rare')])


## 5. Manifest main pack

In [5]:
manifest_main = ManifestPack(
    key="livio-scaled-invasion",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

print(manifest_main)

ManifestPack(key='livio-scaled-invasion', source=ManifestPackSource(provider='ygoprodeck', external_id='24116'), opening_policy=None)


## 6. Assemblaggio main pack

In [6]:
main_pack = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert isinstance(main_pack, PackDefinition)
print(main_pack)

PackDefinition(key='livio-scaled-invasion', name='Scaled Invasion', description='Offline assembly test pack', type=<PackType.PERSONAL_MAIN: 'personal_main'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=100, booster_size=7, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key=None, cards=[Card(name='Card N', ydk_id=1001, source_rarity='Common', master_duel_id=501, rarity=<Rarity.N: 'N'>), Card(name='Card R', ydk_id=1002, source_rarity='Rare', master_duel_id=502, rarity=<Rarity.R: 'R'>), Card(name='Card SR', ydk_id=1003, source_rarity='Super Rare', master_duel_id=503, rarity=<Rarity.SR: 'SR'>), Card(name='Card UR', ydk_id=1004, source_rarity='Ultra Rare', master_duel_id=504, rarity=<Rarity.UR: 'UR'>), Card(name='Card Secret', ydk_id=1005, source_rarity='Secret Rare', master_duel_id=505, rarity=<Rarity.N: 'N'>)])


## 7. Metadata main pack

In [7]:
assert main_pack.key == "livio-scaled-invasion"
assert main_pack.name == "Scaled Invasion"
assert main_pack.description == "Offline assembly test pack"

assert main_pack.type is PackType.PERSONAL_MAIN
assert main_pack.owner_key == "livio"
assert main_pack.parent_pack_key is None
assert main_pack.opening_policy_key == "configuration-a"

assert main_pack.source.provider is SourceProvider.YGOPRODECK
assert main_pack.source.external_id == "24116"

print("Main metadata OK")

Main metadata OK


## 8. Regola main pack

In [8]:
rule = get_pack_rule(PackType.PERSONAL_MAIN)

assert main_pack.expected_card_count == 100
assert main_pack.booster_size == 7

assert rule.rarity_composition.normal == 50
assert rule.rarity_composition.rare == 25
assert rule.rarity_composition.super_rare == 15
assert rule.rarity_composition.ultra_rare == 10
assert rule.rarity_composition.total == 100

print(rule)

PackRule(expected_card_count=100, booster_size=7, rarity_composition=RarityComposition(normal=50, rare=25, super_rare=15, ultra_rare=10))


## 9. Main: Secret Rare -> N

In [9]:
secret_card = next(
    card for card in main_pack.cards
    if card.source_rarity == "Secret Rare"
)

assert secret_card.rarity is Rarity.N

assert [card.rarity for card in main_pack.cards] == [
    Rarity.N,
    Rarity.R,
    Rarity.SR,
    Rarity.UR,
    Rarity.N,
]

print(secret_card)

Card(name='Card Secret', ydk_id=1005, source_rarity='Secret Rare', master_duel_id=505, rarity=<Rarity.N: 'N'>)


## 10. Master Duel IDs

In [10]:
assert [card.master_duel_id for card in main_pack.cards] == [
    501, 502, 503, 504, 505
]

print("Master Duel IDs OK")

Master Duel IDs OK


## 11. Opening-policy override

In [11]:
manifest_override = deepcopy(manifest_main)
manifest_override.opening_policy = "configuration-b"

pack = assemble_pack_definition(
    manifest_pack=manifest_override,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert pack.opening_policy_key == "configuration-b"
print("Opening policy override OK")

Opening policy override OK


## 12. Mini: Secret Rare -> N

In [12]:
manifest_mini = ManifestPack(
    key="livio-mini-test",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

mini_pack = assemble_pack_definition(
    manifest_pack=manifest_mini,
    source_pack=source_pack,
    pack_type=PackType.PERSONAL_MINI,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
    parent_pack_key="livio-scaled-invasion",
)

assert mini_pack.type is PackType.PERSONAL_MINI
assert mini_pack.owner_key == "livio"
assert mini_pack.parent_pack_key == "livio-scaled-invasion"
assert mini_pack.expected_card_count == 20
assert mini_pack.booster_size == 3

secret_card = next(
    card for card in mini_pack.cards
    if card.source_rarity == "Secret Rare"
)

assert secret_card.rarity is Rarity.N

print(secret_card)

Card(name='Card Secret', ydk_id=1005, source_rarity='Secret Rare', master_duel_id=505, rarity=<Rarity.N: 'N'>)


## 13. Shared staples: Secret Rare -> UR

In [13]:
manifest_shared = ManifestSharedPack(
    key="shared-staples",
    type="shared_staples",
    source=ManifestPackSource(
        provider="ygoprodeck",
        external_id="24116",
    ),
    opening_policy=None,
)

shared_pack = assemble_pack_definition(
    manifest_pack=manifest_shared,
    source_pack=source_pack,
    pack_type=PackType.SHARED_STAPLES,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
)

assert shared_pack.type is PackType.SHARED_STAPLES
assert shared_pack.owner_key is None
assert shared_pack.parent_pack_key is None
assert shared_pack.expected_card_count == 200
assert shared_pack.booster_size == 11

secret_card = next(
    card for card in shared_pack.cards
    if card.source_rarity == "Secret Rare"
)

assert secret_card.rarity is Rarity.UR

print(secret_card)

Card(name='Card Secret', ydk_id=1005, source_rarity='Secret Rare', master_duel_id=505, rarity=<Rarity.UR: 'UR'>)


## 14. Nuova PACK_RULE shared staples

In [14]:
shared_rule = get_pack_rule(PackType.SHARED_STAPLES)
composition = shared_rule.rarity_composition

assert composition.normal == 100
assert composition.rare == 60
assert composition.super_rare == 20
assert composition.ultra_rare == 20
assert composition.total == 200

print(shared_rule)

PackRule(expected_card_count=200, booster_size=11, rarity_composition=RarityComposition(normal=100, rare=60, super_rare=20, ultra_rare=20))


## 15. Carta YDK irrisolta

In [15]:
source_with_missing_id = deepcopy(source_pack)
source_with_missing_id.cards.append(
    YgoProDeckPackCard(
        name="Unresolved Card",
        ydk_id=99999999,
        card_type="Effect Monster",
        source_rarity="Common",
    )
)

pack = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_with_missing_id,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

unresolved = [
    card for card in pack.cards
    if card.master_duel_id is None
]

assert len(unresolved) == 1
assert unresolved[0].ydk_id == 99999999
assert unresolved[0].rarity is Rarity.N

print(unresolved[0])

Card(name='Unresolved Card', ydk_id=99999999, source_rarity='Common', master_duel_id=None, rarity=<Rarity.N: 'N'>)


## 16. External ID incoerente

In [16]:
wrong_source = deepcopy(source_pack)
wrong_source.external_id = "99999"

try:
    assemble_pack_definition(
        manifest_pack=manifest_main,
        source_pack=wrong_source,
        pack_type=PackType.PERSONAL_MAIN,
        default_opening_policy_key="configuration-a",
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        owner_key="livio",
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("External ID incoerente non rifiutato.")

Expected ValueError: Source pack external ID does not match manifest pack source external ID.


## 17. Rarità sorgente sconosciuta

In [17]:
source_with_unknown_rarity = deepcopy(source_pack)
source_with_unknown_rarity.cards[0].source_rarity = "Collector's Rare"

try:
    assemble_pack_definition(
        manifest_pack=manifest_main,
        source_pack=source_with_unknown_rarity,
        pack_type=PackType.PERSONAL_MAIN,
        default_opening_policy_key="configuration-a",
        rarity_mapping=rarity_mapping,
        ydk_mapping=ydk_mapping,
        owner_key="livio",
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("Rarità sconosciuta non rifiutata.")

Expected ValueError: Unsupported source rarity "Collector's Rare" for pack type <PackType.PERSONAL_MAIN: 'personal_main'>.


## 18. DTO sorgente non modificato

In [18]:
source_copy = deepcopy(source_pack)

assembled = assemble_pack_definition(
    manifest_pack=manifest_main,
    source_pack=source_copy,
    pack_type=PackType.PERSONAL_MAIN,
    default_opening_policy_key="configuration-a",
    rarity_mapping=rarity_mapping,
    ydk_mapping=ydk_mapping,
    owner_key="livio",
)

assert source_copy.cards[-1].source_rarity == "Secret Rare"
assert not hasattr(source_copy.cards[-1], "rarity")
assert not hasattr(source_copy.cards[-1], "master_duel_id")

assert assembled.cards[-1] is not source_copy.cards[-1]

print("Source DTO remains unchanged")

Source DTO remains unchanged
